# THESIS-004 — Data Collection Scripts
**Story Points:** 3 | **Status:** DONE

All 5 collection scripts are implemented. Tested end-to-end with L1 runs.


## Acceptance Criteria
- [x] `collect_vm_metrics.py` — CPU/memory/process CSV
- [x] `collect_k8s_metrics.sh` — pod-level CPU/memory via kubectl top
- [x] `collect_pod_timing.py` — pod lifecycle timestamps
- [x] `export_dagster_runs.py` — Dagster PostgreSQL -> CSV
- [x] `trigger_dagster_runs.py` — launches N concurrent runs
- [x] All scripts tested end-to-end
- [ ] Graceful SIGTERM shutdown verified for collector scripts

## Verify script headers

In [ ]:
import subprocess
scripts = [
    '../scripts/collect_vm_metrics.py',
    '../scripts/collect_k8s_metrics.sh',
    '../scripts/collect_pod_timing.py',
    '../scripts/export_dagster_runs.py',
    '../scripts/trigger_dagster_runs.py',
]
for s in scripts:
    r = subprocess.run(['head', '-5', s], capture_output=True, text=True)
    print(f'-- {s.split("/")[-1]} --')
    print(r.stdout)

## Test collect_k8s_metrics.sh (5-second sample)

In [ ]:
import subprocess, tempfile, os, time
out_csv = tempfile.mktemp(suffix='.csv')
proc = subprocess.Popen(
    ['bash', '../scripts/collect_k8s_metrics.sh', '--output', out_csv],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(6)
proc.terminate()
proc.wait()
r = subprocess.run(['head', '-5', out_csv], capture_output=True, text=True)
print(r.stdout if os.path.exists(out_csv) else 'No output file')
os.unlink(out_csv) if os.path.exists(out_csv) else None

## Test collect_pod_timing.py

In [ ]:
import subprocess, tempfile, os
out_csv = tempfile.mktemp(suffix='.csv')
r = subprocess.run(
    ['python3', '../scripts/collect_pod_timing.py',
     '--namespace', 'dagster', '--output', out_csv],
    capture_output=True, text=True, timeout=30
)
print(r.stdout[:500])
if os.path.exists(out_csv):
    r2 = subprocess.run(['head', '-5', out_csv], capture_output=True, text=True)
    print(r2.stdout)
    os.unlink(out_csv)
else:
    print('No CSV output (expected if no run pods exist)')